# Firstly , Load the Dataset

In [2]:
with open('tiny_textbooks_clean.txt','r',encoding='utf-8') as f:
  raw_text=f.read()

print('total no. of characters present in dataset is ',len(raw_text))
print(raw_text[:99])

total no. of characters present in dataset is  226294848
deficit financing. Also found in: Dictionary, Thesaurus, Wikipedia.. deficit financing. The sale of


#  There are around 22,62,94,848 characters present in our tiny_textbooks_clean.txt file

# Split our Dataset into Training and Validation Dataset

In [3]:
import os
import torch
from torch.utils.data import Dataset, DataLoader


# Production Standard: 90% Train, 10% Validation split calculation
split_idx = int(len(raw_text) * 0.90)

train_text = raw_text[:split_idx]
val_text = raw_text[split_idx:]

print("📊 Dataset Slicing Metrics:")
print(f"   -> Total Global Characters: {len(raw_text):,}")
print(f"   -> Training Characters     : {len(train_text):,}")
print(f"   -> Validation Characters   : {len(val_text):,}\n")

📊 Dataset Slicing Metrics:
   -> Total Global Characters: 226,294,848
   -> Training Characters     : 203,665,363
   -> Validation Characters   : 22,629,485



# Now, Use BPE Tokenizer in order to tokenize our entire Training Corpus

In [4]:
# to use BPE tokenizer, we need to import a module named as tiktoken
import tiktoken
tokenizer=tiktoken.get_encoding('gpt2')

sample_text='hello sahil! how are you?'
integers=tokenizer.encode(sample_text,allowed_special={'<|endoftext|>'})
print(integers)

[31373, 473, 71, 346, 0, 703, 389, 345, 30]


In [5]:
text=''.join(tokenizer.decode(integers))
text

'hello sahil! how are you?'

In [6]:
# Now, apply this BPE Tokenizer on our train corpus which is stored in train_text
token_ids=tokenizer.encode(train_text,allowed_special={'<|endoftext|>'})

In [7]:
train_token_ids=token_ids

In [8]:
print('first 100 token ids are ->',train_token_ids[:100])

first 100 token ids are -> [4299, 3628, 15435, 13, 4418, 1043, 287, 25, 28261, 11, 383, 82, 22302, 11, 15312, 492, 11807, 15435, 13, 383, 5466, 286, 5057, 16145, 287, 1502, 284, 9604, 22895, 326, 389, 287, 6992, 286, 3739, 13, 23904, 11, 11807, 15435, 318, 5625, 284, 1230, 9604, 780, 3739, 11, 7997, 416, 1687, 13089, 290, 6642, 11, 318, 1690, 23485, 284, 1414, 9307, 13, 1081, 351, 32153, 2890, 262, 5057, 11, 11807, 15435, 7584, 18644, 3833, 319, 1393, 3965, 780, 1230, 5057, 16145, 9320, 351, 2839, 16145, 329, 3614, 3139, 13, 198, 50256, 198, 45638, 5890, 11, 3334, 1322, 784, 406, 32894]


In [9]:
# Tiktoken standard code for viewing individual tokens
print('first 100 textual tokens via BPE tokenizer are:\n',
      [tokenizer.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in train_token_ids[:100]])

first 100 textual tokens via BPE tokenizer are:
 ['def', 'icit', ' financing', '.', ' Also', ' found', ' in', ':', ' Dictionary', ',', ' The', 's', 'aurus', ',', ' Wikipedia', '..', ' deficit', ' financing', '.', ' The', ' sale', ' of', ' debt', ' securities', ' in', ' order', ' to', ' finance', ' expenditures', ' that', ' are', ' in', ' excess', ' of', ' income', '.', ' Generally', ',', ' deficit', ' financing', ' is', ' applied', ' to', ' government', ' finance', ' because', ' income', ',', ' represented', ' by', ' tax', ' revenues', ' and', ' fees', ',', ' is', ' often', ' unavailable', ' to', ' pay', ' expenses', '.', ' As', ' with', ' monet', 'izing', ' the', ' debt', ',', ' deficit', ' financing', ' puts', ' upward', ' pressure', ' on', ' interest', ' rates', ' because', ' government', ' debt', ' securities', ' compete', ' with', ' private', ' securities', ' for', ' limited', ' capital', '.', '\n', '<|endoftext|>', '\n', 'Technical', ' Director', ',', ' High', 'ways', ' –', ' L',

# Now, we will see that in how many tokens our training corpus was tokenized

In [10]:
print('our training corpus is tokenized into ',len(train_token_ids),'no. of tokens (including repeated tokens)')

our training corpus is tokenized into  45560296 no. of tokens (including repeated tokens)


# Now let's see the unique tokens out of 4,55,60,296 tokens

In [11]:
print('the total no. of unique tokens in which our train corpus was tokenized is ',len(set(train_token_ids)))

the total no. of unique tokens in which our train corpus was tokenized is  49822


# Now, apply Tokenization on our Validation data as well

In [12]:
# Now, apply this BPE Tokenizer on our validationn corpus which is stored in val_text
val_token_ids=tokenizer.encode(val_text,allowed_special={'<|endoftext|>'})

In [13]:
print('first 100 token ids are ->',val_token_ids[:100])

first 100 token ids are -> [1691, 5694, 8315, 274, 1978, 290, 345, 447, 247, 297, 423, 2818, 6737, 6082, 290, 1630, 49052, 287, 257, 2746, 492, 198, 50256, 198, 43, 945, 19853, 268, 198, 198, 43, 945, 13002, 50007, 24172, 35906, 19853, 268, 357, 6286, 2310, 3035, 19342, 287, 39636, 8, 318, 257, 311, 6557, 11632, 12, 21991, 20684, 58, 16, 60, 324, 5388, 81, 290, 10099, 11, 5863, 329, 465, 39180, 602, 290, 736, 41291, 13604, 1756, 287, 11859, 22775, 13, 554, 1948, 11, 339, 2627, 2592, 1900, 329, 465, 33834, 12, 71, 14132, 5296, 287, 7840, 3340, 11, 543, 373, 18976, 290, 12395]


In [14]:
# Tiktoken standard code for viewing individual tokens
print('first 100 textual tokens via BPE tokenizer are:\n',
      [tokenizer.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in val_token_ids[:100]])

first 100 textual tokens via BPE tokenizer are:
 ['eral', ' Central', ' Box', 'es', ' together', ' and', ' you', '�', '�', 'll', ' have', ' perfect', ' signal', ' distribution', ' and', ' control', ' redundancy', ' in', ' a', ' model', '..', '\n', '<|endoftext|>', '\n', 'L', 'ars', ' Mons', 'en', '\n', '\n', 'L', 'ars', ' Thor', 'bj', 'ø', 'rn', ' Mons', 'en', ' (', 'born', ' 21', ' April', ' 1963', ' in', ' Oslo', ')', ' is', ' a', ' S', 'á', 'mi', '-', 'Nor', 'wegian', '[', '1', ']', 'ad', 'venture', 'r', ' and', ' journalist', ',', ' famous', ' for', ' his', ' explor', 'ations', ' and', ' back', 'packing', ' exped', 'itions', ' in', ' harsh', ' wilderness', '.', ' In', ' particular', ',', ' he', ' became', ' especially', ' known', ' for', ' his', ' thru', '-', 'h', 'iking', ' trip', ' in', ' northern', ' Canada', ',', ' which', ' was', ' filmed', ' and', ' documented']


#  Now, we will see that in how many tokens our validation corpus was tokenized

In [15]:
print('our training corpus is tokenized into ',len(val_token_ids),'no. of tokens (including repeated tokens)')

our training corpus is tokenized into  5068902 no. of tokens (including repeated tokens)


#  Now let's see the unique tokens out of 50,68,902 tokens

In [16]:
print('the total no. of unique tokens in which our validation corpus was tokenized is ',len(set(val_token_ids)))

the total no. of unique tokens in which our validation corpus was tokenized is  48028


# Till Now, we have Successfully Performed Tokenization on our both Training and Validation Corpus

# **Now, Next Phase is to make Input-Output Pairs on our Training Corpus**

In [17]:
context_length=4 # model is supposed to see the 4 tokens as an input to predict the next token and for illustration purpose, we are taking such a small context length
Input=train_token_ids[:context_length]
Output=train_token_ids[1:context_length+1]
print('Input is -->',Input)
print('Output is -->',Output)

Input is --> [4299, 3628, 15435, 13]
Output is --> [3628, 15435, 13, 4418]


In [18]:
for i in range(1,context_length+1):
  print(train_token_ids[:i],"-->",train_token_ids[i])

[4299] --> 3628
[4299, 3628] --> 15435
[4299, 3628, 15435] --> 13
[4299, 3628, 15435, 13] --> 4418


In [19]:
for i in range(1,context_length+1):
  Input,Output=train_token_ids[:i],[train_token_ids[i]]
  print(tokenizer.decode(Input),"-->",tokenizer.decode(Output))

def --> icit
deficit -->  financing
deficit financing --> .
deficit financing. -->  Also


# See , the above given code was just for illustration purpose, we need to create a Dataloader that would efficiently handle making I-O pairs for our entire training corpus

In [20]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []

        # Tokenize the entire text safely handling end-of-text special tokens
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the dataset into overlapping sequences
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [21]:
def create_dataloaderv1(text,batch_size=10,max_length=1024,stride=1000,shuffle=True,drop_last=True,num_workers=0):

  # initialize the tokenizer
  tokenizer=tiktoken.get_encoding("gpt2")

  # Create Dataset
  dataset=GPTDatasetV1(text,tokenizer,max_length,stride)

  # Create Dataloader
  dataloader=DataLoader(
      dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers
  )
  return dataloader

In [22]:
import torch
dataloader=create_dataloaderv1(train_text,batch_size=10,max_length=1024,stride=1000)
data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)


[tensor([[   13,  1320, 15787,  ...,   714,  1912,   319],
        [  257, 20128,  6464,  ...,   262,  4217,   314],
        [   25,   198,   198,  ...,  8758, 18537,   815],
        ...,
        [  843,    72,  5929,  ...,  8233,   666,  1719],
        [ 6586,   416,   262,  ..., 13613,    13,  1367],
        [ 6872,   503,   262,  ...,  5127,  2233,   284]]), tensor([[ 1320, 15787, 33181,  ...,  1912,   319,   616],
        [20128,  6464,  7604,  ...,  4217,   314,  3535],
        [  198,   198,   447,  ..., 18537,   815,   307],
        ...,
        [   72,  5929, 47827,  ...,   666,  1719,   281],
        [  416,   262,   717,  ...,    13,  1367,    14],
        [  503,   262,  1811,  ...,  2233,   284, 23476]])]


In [23]:
input_sequence,target_sequence=first_batch
print('input sequence is ->')
print(input_sequence.shape)
print(input_sequence)
print(100*'*')
print('output sequence is ->')
print(target_sequence.shape)
print(target_sequence)



input sequence is ->
torch.Size([10, 1024])
tensor([[   13,  1320, 15787,  ...,   714,  1912,   319],
        [  257, 20128,  6464,  ...,   262,  4217,   314],
        [   25,   198,   198,  ...,  8758, 18537,   815],
        ...,
        [  843,    72,  5929,  ...,  8233,   666,  1719],
        [ 6586,   416,   262,  ..., 13613,    13,  1367],
        [ 6872,   503,   262,  ...,  5127,  2233,   284]])
****************************************************************************************************
output sequence is ->
torch.Size([10, 1024])
tensor([[ 1320, 15787, 33181,  ...,  1912,   319,   616],
        [20128,  6464,  7604,  ...,  4217,   314,  3535],
        [  198,   198,   447,  ..., 18537,   815,   307],
        ...,
        [   72,  5929, 47827,  ...,   666,  1719,   281],
        [  416,   262,   717,  ...,    13,  1367,    14],
        [  503,   262,  1811,  ...,  2233,   284, 23476]])


# Till Now, we have Broke down our Training corpus into Multiple Batches. each batch has 10 Input Sequence, and Each Input Sequence has 1024 Tokens as per the Context Length

# **Now, We need To create Token Embedding**

In [24]:
vocab_size=50257
output_dimension=768
torch.manual_seed(123)
embedding_layer= torch.nn.Embedding(vocab_size,output_dimension)

In [25]:
print('dimension of our Token Embedding is ',embedding_layer.weight.shape)

dimension of our Token Embedding is  torch.Size([50257, 768])


In [26]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 3.3737e-01, -1.7778e-01, -3.0353e-01,  ..., -3.1812e-01,
         -1.3936e+00,  5.2262e-01],
        [ 2.5787e-01,  3.4197e-01, -8.1678e-01,  ..., -4.0981e-01,
          4.9785e-01, -3.7207e-01],
        [ 7.9574e-01,  5.3501e-01,  9.4275e-01,  ..., -1.0749e+00,
          9.5492e-02, -1.4138e+00],
        ...,
        [-7.1278e-01, -5.0190e-01,  1.4119e+00,  ..., -1.4979e-01,
         -4.8977e-01, -1.0620e+00],
        [ 2.0646e+00,  1.1190e+00,  3.8486e-01,  ..., -7.2015e-01,
         -5.5703e-01,  9.8639e-01],
        [ 1.1364e-03, -7.5320e-01, -1.7924e-01,  ..., -3.2443e-01,
          2.6055e-01,  5.8885e-01]], requires_grad=True)


# **Now, we need to make the token embedding for our current batch**

In [27]:
embedding_layer.weight[input_sequence].shape

torch.Size([10, 1024, 768])

In [28]:
print('our token embedding for our current batch is given below ')

our token embedding for our current batch is given below 


In [29]:
token_embeddings=embedding_layer.weight[input_sequence]

In [30]:
token_embeddings

tensor([[[-9.9317e-01, -1.8565e-01, -9.2415e-01,  ..., -1.0024e+00,
           8.7165e-01,  2.4830e-01],
         [ 1.0547e+00, -2.2678e-01, -9.1365e-01,  ..., -6.5796e-01,
           6.0148e-02,  1.6396e-01],
         [-3.3350e-01, -1.0366e+00, -2.1202e-01,  ...,  6.9470e-01,
          -1.2268e+00, -4.3580e-01],
         ...,
         [-3.4783e-01,  1.6271e+00, -7.2383e-01,  ..., -6.4276e-01,
          -1.2356e+00, -1.3985e+00],
         [-7.4480e-01, -3.1224e-01,  9.7517e-01,  ...,  2.3229e-01,
          -2.4664e+00, -1.3448e+00],
         [ 5.0863e-01, -4.1173e-01, -1.4217e-02,  ..., -9.2957e-02,
           8.7662e-01,  9.7499e-01]],

        [[ 1.2758e+00, -2.8963e-01,  9.5391e-01,  ...,  3.4050e-01,
          -1.0636e+00, -6.7363e-02],
         [ 2.2004e-01,  1.3413e+00, -2.0437e+00,  ...,  1.4131e+00,
           1.2613e+00, -2.5756e-01],
         [-6.4655e-01, -6.4765e-01, -3.7212e-01,  ...,  4.5584e-01,
          -9.3929e-01,  1.1205e+00],
         ...,
         [-2.0215e+00, -2

# **Now, we need to create positional embedding of dimension [context length , 768]**

In [31]:
context_length=1024

In [32]:
pos_embedding_layer=torch.nn.Embedding(context_length,output_dimension)

In [33]:
pos_embedding_layer.weight.shape

torch.Size([1024, 768])

In [34]:
pos_embeddings=pos_embedding_layer(torch.arange(context_length))
pos_embeddings.shape

torch.Size([1024, 768])

# **Now, we need to create Input Embedding by just adding token embeddings and positional embeddings**

In [45]:
input_embeddings=pos_embeddings + token_embeddings
print(input_embeddings.shape)

torch.Size([10, 1024, 768])


In [46]:
print('our input embeddings for our current batch is given below ->')
input_embeddings

our input embeddings for our current batch is given below ->


tensor([[[-0.1163,  0.0694, -0.0800,  ..., -2.0377,  2.1801,  2.0440],
         [ 0.0518, -0.1273,  0.3323,  ...,  0.8874, -0.0525, -1.3558],
         [ 0.9982, -0.2806,  0.6957,  ...,  0.7777,  0.6068, -2.6583],
         ...,
         [-0.4533,  0.4330, -1.8710,  ..., -2.0971, -0.9438, -1.9467],
         [-0.5230, -0.6455, -0.3623,  ..., -0.1957, -2.5471,  0.4335],
         [ 1.6163, -0.3184,  0.8252,  ..., -1.5686,  2.0579,  3.3421]],

        [[ 2.1527, -0.0346,  1.7980,  ..., -0.6949,  0.2449,  1.7284],
         [-0.7829,  1.4407, -0.7978,  ...,  2.9584,  1.1487, -1.7773],
         [ 0.6851,  0.1084,  0.5356,  ...,  0.5388,  0.8943, -1.1021],
         ...,
         [-2.1269, -1.4109, -0.2898,  ..., -1.4566,  0.3633,  0.8806],
         [-0.4616, -1.2622, -1.1351,  ...,  1.0165,  0.1011,  0.6169],
         [-0.5230,  0.0269,  0.7196,  ..., -2.0488,  2.0089,  2.4476]],

        [[ 3.7797,  0.1523,  1.1760,  ..., -0.2212,  0.6486,  0.9972],
         [ 0.7096,  0.6479,  0.0187,  ...,  1

# ** Note that Till Now, we have Successfully Created the Input Embedding.
# Reminder->kindly note that all the stuff that we've done till now was only for 1 batch, but for now, there's no need to iterate on each batch and perform all such steps as we will do this in pretraining  

# **IMPLEMENTING MULTI-HEAD ATTENTION WITH WEIGHT SPLITS**


In [49]:
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return context_vec

In [58]:
torch.manual_seed(123)
batch_size, context_length, d_in = input_embeddings.shape
d_out = d_in
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(input_embeddings)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 5.2291e-01,  5.0445e-01,  2.7872e-01,  ..., -1.0393e+00,
           3.2695e-01,  2.6530e-01],
         [ 5.4713e-01,  1.3640e-01,  6.7064e-01,  ..., -1.4184e-01,
           1.9809e-01,  1.7277e-01],
         [ 3.0660e-01,  8.5145e-01,  2.3121e-01,  ..., -1.0922e-01,
          -1.2017e-02,  4.5658e-02],
         ...,
         [-1.5998e-02, -1.0186e-01, -8.9326e-02,  ..., -7.7789e-02,
          -3.1294e-02,  4.2103e-03],
         [-9.7749e-02, -5.9146e-02, -9.0337e-02,  ..., -3.0428e-02,
          -4.6813e-02,  2.4428e-02],
         [-8.0401e-02, -1.4016e-01, -1.4585e-01,  ..., -1.1839e-02,
          -5.0591e-02, -2.1111e-02]],

        [[ 7.4677e-01, -1.3062e-01,  8.5569e-01,  ...,  1.0391e-01,
          -6.9513e-01,  8.2024e-01],
         [ 2.8116e-01,  9.8076e-01,  8.3384e-01,  ...,  2.2012e-01,
          -4.4808e-01,  1.4846e-01],
         [ 4.1042e-01,  6.6755e-01,  6.7046e-01,  ...,  4.3375e-01,
          -4.1208e-01, -4.7905e-01],
         ...,
         [ 5.5422e-03, -6

# **Kindly Note that the above Execution was just for the Purpose of Demonstration Purpose that how your Contextual Vectors for the respective tokens looks like, but we will do actual work in Pretraining Phase. The above given work was not the actual work , we will do Pretraining Soon**